# Readers

> Read sets from CSV, TSV and Excel files.

In [ ]:
#| default_exp readers

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

`load_sets` accepts the two layouts people already keep their hits in, and works out which one it has been given:

1. **A membership table**: one row per item, with a column called `gene` (or `id`, `name`, `protein`, `prey`, `item`, `symbol`), a yes/no column for each set, and optionally a `category` column (plus `color`).
2. **One column per set**: each set's name as a column header, with its items listed underneath. Blank cells are ignored.

Categories (for example GO terms) colour the item names. They can come from the membership table's `category` column, from a separate file passed as `categories_path` (columns `gene, category[, color]`), or from a sheet called `categories` in the same Excel workbook.

In [ ]:
#| export
import csv
import math
from pathlib import Path

import pandas as pd

In [ ]:
#| export
TRUTHY = {"1", "1.0", "y", "yes", "true", "t", "x", "+", "✓"}
ID_COLS = ("gene", "genes", "id", "name", "protein", "prey", "item", "symbol")
EXCEL_SUFFIXES = (".xlsx", ".xlsm", ".xls")

def _sniff_sep(path, default):
    "Comma, tab or semicolon, whichever the file uses; `default` if it has only one column."
    with open(path, encoding="utf-8-sig", newline="") as f:
        sample = f.read(64 * 1024)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",\t;").delimiter
    except csv.Error:
        return default

def _read_table(path, sheet=None):
    path = Path(path)
    suf = path.suffix.lower()
    if suf in EXCEL_SUFFIXES:
        return pd.read_excel(path, sheet_name=sheet if sheet is not None else 0, dtype=str)
    sep = _sniff_sep(path, "\t" if suf in (".tsv", ".tab") else ",")
    return pd.read_csv(path, sep=sep, dtype=str, encoding="utf-8-sig")

def _clean(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return str(x).strip()

In [ ]:
#| export
def load_sets(path, sheet=None, categories_path=None, categories_sheet=None):
    "Read sets from `path`. Returns (sets: {name: [items]}, categories: {item: category}, category colours: {category: colour})."
    df = _read_table(path, sheet)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, [c for c in df.columns if not c.lower().startswith("unnamed")]]
    lower = {c.lower(): c for c in df.columns}
    categories, cat_colours = {}, {}

    id_col = next((lower[k] for k in ID_COLS if k in lower), None)
    if id_col is not None:                                   # membership table
        cat_col = lower.get("category") or lower.get("group") or lower.get("class")
        col_col = lower.get("color") or lower.get("colour")
        set_cols = [c for c in df.columns if c not in (id_col, cat_col, col_col)]
        sets = {s: [] for s in set_cols}
        for _, row in df.iterrows():
            item = _clean(row[id_col])
            if not item:
                continue
            for s in set_cols:
                if _clean(row[s]).lower() in TRUTHY:
                    sets[s].append(item)
            if cat_col and _clean(row[cat_col]):
                categories[item] = _clean(row[cat_col])
                if col_col and _clean(row[col_col]):
                    cat_colours[categories[item]] = _clean(row[col_col])
    else:                                                    # one column per set
        sets = {c: [v for v in (_clean(x) for x in df[c]) if v] for c in df.columns}

    # categories from a separate file, or from a 'categories' sheet of the same workbook
    cpath, csheet = categories_path, categories_sheet
    if cpath is None and Path(path).suffix.lower() in EXCEL_SUFFIXES:
        names = pd.ExcelFile(path).sheet_names
        hit = next((n for n in names if n.strip().lower() in ("categories", "category")), None)
        if hit:
            cpath, csheet = path, hit
    if cpath is not None:
        cdf = _read_table(cpath, csheet)
        cl = {str(c).strip().lower(): c for c in cdf.columns}
        gcol = next((cl[k] for k in ID_COLS if k in cl), cdf.columns[0])
        ccol = cl.get("category", cdf.columns[1])
        kcol = cl.get("color") or cl.get("colour")
        for _, row in cdf.iterrows():
            g, c = _clean(row[gcol]), _clean(row[ccol])
            if g and c:
                categories[g] = c
                if kcol and _clean(row[kcol]):
                    cat_colours[c] = _clean(row[kcol])

    sets = {k: list(dict.fromkeys(v)) for k, v in sets.items()}   # drop duplicates, keep order
    if not 1 <= len(sets) <= 3:
        raise ValueError(f"Need 1 to 3 sets, found {len(sets)}: {list(sets)}")
    return sets, categories, cat_colours

## Examples

A membership table, with categories:

In [ ]:
tmp = Path(tempfile.mkdtemp())
(tmp/"membership.csv").write_text(
    "gene,DDX6,LSM14A,category\n"
    "EDC4,1,1,Decapping\n"
    "PATL1,yes,,\n"
    "CNOT1,x,x,CCR4-NOT\n"
    "LSM14B,,1,\n"
    "PATL1,1,,\n")
sets, cats, cols = load_sets(tmp/"membership.csv")
test_eq(sets, {"DDX6": ["EDC4", "PATL1", "CNOT1"], "LSM14A": ["EDC4", "CNOT1", "LSM14B"]})
test_eq(cats, {"EDC4": "Decapping", "CNOT1": "CCR4-NOT"})
sets

{'DDX6': ['EDC4', 'PATL1', 'CNOT1'], 'LSM14A': ['EDC4', 'CNOT1', 'LSM14B']}

One column per set, as a tab-separated file:

In [ ]:
(tmp/"wide.tsv").write_text("DDX6\tLSM14A\tEIF4ENIF1\nEDC4\tEDC4\tLSM14A\nPATL1\tLSM14B\t\n\tCNOT1\t\n")
sets, cats, cols = load_sets(tmp/"wide.tsv")
test_eq(sets, {"DDX6": ["EDC4", "PATL1"], "LSM14A": ["EDC4", "LSM14B", "CNOT1"], "EIF4ENIF1": ["LSM14A"]})

A file with a single column is one set. (The prototype let pandas guess the separator from any character, and it split a column called `Bait1` at the letter "t".) Semicolon-separated files, common where the decimal mark is a comma, work too:

In [ ]:
(tmp/"one.csv").write_text("Bait1\nA_00\nA_01\nBTZ\n")
test_eq(load_sets(tmp/"one.csv")[0], {"Bait1": ["A_00", "A_01", "BTZ"]})
(tmp/"semi.csv").write_text("gene;Bait1;Bait2\nEDC4;1;1\nPATL1;1;0\n")
test_eq(load_sets(tmp/"semi.csv")[0], {"Bait1": ["EDC4", "PATL1"], "Bait2": ["EDC4"]})

An Excel workbook with the sets on one sheet and a `categories` sheet with colours:

In [ ]:
xl = tmp/"sets.xlsx"
with pd.ExcelWriter(xl) as w:
    pd.DataFrame({"DDX6": ["EDC4", "PATL1"], "LSM14A": ["EDC4", "LSM14B"]}).to_excel(w, sheet_name="sets", index=False)
    pd.DataFrame({"gene": ["EDC4"], "category": ["Decapping"], "color": ["#C8102E"]}).to_excel(w, sheet_name="categories", index=False)
sets, cats, cols = load_sets(xl, sheet="sets")
test_eq(sets, {"DDX6": ["EDC4", "PATL1"], "LSM14A": ["EDC4", "LSM14B"]})
test_eq((cats, cols), ({"EDC4": "Decapping"}, {"Decapping": "#C8102E"}))

More than three sets is an error, because the diagram can't show them (see [why it stops at three](https://adamcc.github.io/itemiset/#three)):

In [ ]:
(tmp/"four.csv").write_text("A,B,C,D\nx,y,z,w\n")
test_fail(lambda: load_sets(tmp/"four.csv"), contains="Need 1 to 3 sets")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()